# Telco Customer Churn

Runs the full pipeline on **Google Colab**: ML task, data collection, integration, cleaning,
transformation, visualization and ETL, then the model. Each cell calls one step from `src/`. Run it with **Runtime → Run all**. The full run takes about
10 minutes; set `RUN_EXPERIMENTS = False` below to load the saved experiment results instead
(about 2 minutes).

## 0. Setup

In [ ]:
RUN_EXPERIMENTS = True   # False = load the saved experiment tables instead of recomputing them
REPO = "Peter7168/comp333-telco-churn"

import os, subprocess, sys
from getpass import getpass

# Clone the repo unless already inside it.
if not os.path.exists("src/pipeline.py"):
    folder = REPO.split("/")[1]
    if not os.path.exists(folder):
        url = f"https://github.com/{REPO}.git"
        cloned = subprocess.run(["git", "clone", "-q", url], capture_output=True).returncode == 0
        if not cloned:
            # Private repo: ask for a read-only GitHub token.
            token = getpass("GitHub token (read access to the repo): ")
            url = f"https://{token}@github.com/{REPO}.git"
            subprocess.run(["git", "clone", "-q", url], check=True)
    os.chdir(folder)
print("working in", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import replace
from IPython.display import Image, display

from src import (dependencies, entity_resolution, etl, experiments, fe_tools, figures, ingest,
                 integrate, lake, train)
from src.clean import NULL_STRATEGIES, OUTLIER_STRATEGIES, null_report, outlier_report
from src.config import TABLES, FIGURES, TARGET
from src.experiments import FINAL, run_grid, split_dev_test, stage_recipes, run_stress
from src.transform import ENCODERS, NORMALIZERS, SCALERS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("storage:", "Delta Lake" if lake.HAS_DELTA else "CSV fallback")


def grid(name, label, recipes):
    """Run one experiment grid on the dev set, or load the saved results."""
    if RUN_EXPERIMENTS:
        table = run_grid(dev, name, label, recipes)
    else:
        table = pd.read_csv(TABLES / f"exp_{name}.csv")
    return table[[label, "f1", "f1_std", "roc_auc", "accuracy", "test_rows"]].round(3)

## 1. Topic and ML task
We predict whether a customer of a telecom company will **cancel their service (churn)**.
It is a **binary classification** task: the answer is Yes (1) or No (0).
We score models with **F1 on the churn class**, because only about 1 in 4 customers churn and
plain accuracy would reward a model that never predicts churn.

## 2. Data collection and storage
Two IBM sources describe the same 7,043 customers:
* **Source 1** (IBM GitHub, CSV): one flat file split into 3 tables.
* **Source 2** (IBM Cognos 11.1.3, Excel): 5 files with demographics, location, zip-code
  population, service usage and status.

We load all 8 files into the **bronze** layer of the **Delta Lake**.

In [ ]:
stats = ingest.run()

In [ ]:
display(Image(str(FIGURES / "00_erd.png"), width=900))

Delta Lake keeps every version of a table. Below is the version log of the bronze
`customer_profiles` table.

In [ ]:
history = pd.DataFrame(lake.history("bronze", "github_customer_profiles"))
history[[c for c in ["version", "timestamp", "operation"] if c in history]] if len(history) else "no Delta history (CSV fallback)"

## 3. Data integration
Each source names and codes the same facts differently. We map all 74 source columns to one
schema, turn Yes/No into 1/0, join everything on the customer ID (and location to population
on zip code), and log every conflict between the sources. Source 1 wins when they disagree.

In [ ]:
integrated = integrate.run()

In [ ]:
pd.read_csv(TABLES / "schema_mapping.csv").head(15)

In [ ]:
pd.read_csv(TABLES / "integration_conflicts.csv").query("conflicts > 0")

Payment method has no one-to-one mapping between the two sources:

In [ ]:
pd.read_csv(TABLES / "payment_method_crosswalk.csv")

**Entity resolution.** Both sources share the customer ID. To check the link, we also match
them **without** the ID: blocking on six stable attributes, then scoring candidate pairs on 11
others.

In [ ]:
entity_resolution.run();

**Functional dependencies** tested on all rows:

In [ ]:
fds, conditional = dependencies.run(integrated)

## 4. Data cleaning
Every cleaning and transformation option below is learned on training folds only and scored with
**5-fold cross-validation on the development set** (80% of customers). The other 20% stay locked
away as the final test set.

In [ ]:
dev, _ = split_dev_test(integrated)
dev = dev.reset_index(drop=True)
print(len(dev), "customers in the development set")

### Null values

In [ ]:
null_report(integrated)

In [ ]:
grid("nulls", "strategy", [(s, replace(FINAL, nulls=s)) for s in NULL_STRATEGIES])

`drop_rows` looks best only because it deletes every customer without an offer, including the
ones it is graded on (see `test_rows`). We fill the blanks instead: "None" for offer and internet
type, 0 for the 11 unbilled customers.

### Outliers

In [ ]:
report, forest_flags = outlier_report(integrated)
print("Isolation Forest flags", forest_flags, "customers")
report[["column", "lower_fence", "upper_fence", "max", "iqr_outliers", "zscore_outliers", "note"]]

In [ ]:
grid("outliers", "strategy", [(s, replace(FINAL, outliers=s)) for s in OUTLIER_STRATEGIES])

### With and without cleaning on dirty data
Our real data is almost clean, so cleaning barely moves the score. Here we damage a copy on
purpose (10% blanks, 3% typing errors) and run the same strategies.

In [ ]:
stress = run_stress(dev) if RUN_EXPERIMENTS else pd.read_csv(TABLES / "exp_stress.csv")
stress[["setup", "f1", "roc_auc", "test_rows"]].round(3)

## 5. Data transformation

In [ ]:
grid("normalizers", "normalizer", [(n, replace(FINAL, normalizer=n)) for n in NORMALIZERS])

Scaling matters for distance-based models (KNN) much more than for logistic regression:

In [ ]:
grid("scalers", "setup", [(f"{s} | {m}", replace(FINAL, scaler=s, model=m))
                          for m in ("logreg", "knn") for s in SCALERS])

In [ ]:
grid("encoders", "encoder", [(e, replace(FINAL, encoder=e)) for e in ENCODERS])

## 6. ETL and visualization
The ETL step writes the cleaned **silver** table and the model-ready **gold** table with our
engineered features (add-on count, price per service, tenure band, ...).

In [ ]:
gold = etl.run(integrated)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["contract", "internet_type", "tenure_band"]):
    (gold.groupby(col)[TARGET].mean() * 100).sort_values().plot.barh(ax=ax, color="#2a78d6")
    ax.set_title(f"Churn rate by {col} (%)")
    ax.set_ylabel("")
plt.tight_layout()

How many features does the model need?

In [ ]:
grid("feature_selection", "features",
     [(f"top {k}", replace(FINAL, top_k=k)) for k in (5, 10, 15, 20, 30)] + [("all", FINAL)])

### Feature-engine
We also used **Feature-engine** (it needs the gold table, so it runs after the ETL step): its `Winsorizer` for IQR capping, its `RareLabelEncoder` to
group rare cities, and `DropCorrelatedFeatures` to find features that repeat each other.

In [ ]:
fe_results = fe_tools.run(integrated)
pd.read_csv(TABLES / "feature_engine_winsorizer.csv")

In [ ]:
pd.read_csv(TABLES / "feature_engine_correlated.csv")

In [ ]:
fe_results[["setup", "f1", "roc_auc", "n_features"]].round(3)

## 7. Model and results
### Performance with and without each stage
Same model (logistic regression) and same folds; each row adds one pipeline stage.

In [ ]:
grid("stages", "stage", stage_recipes())

In [ ]:
grid("models", "model", [(m, replace(FINAL, model=m))
                         for m in ("logreg", "knn", "random_forest", "hist_gb")])

### Final model on the locked test set
The best model is trained on the development set and graded **once** on the 20% test set.

In [ ]:
metrics = train.run(integrated)
m = metrics.iloc[0]
print(f"caught {m.tp} of {m.tp + m.fn} leavers, {m.fp} false alarms")
metrics

In [ ]:
figures.run()
display(Image(str(FIGURES / "01_stages_f1.png"), width=750))
display(Image(str(FIGURES / "07_feature_importance.png"), width=650))

### Scoring customers with the deployed model
The saved model scores any customer. The Streamlit dashboard (`streamlit run dashboard/app.py`,
run on a laptop) uses the same function in its prediction form.

In [ ]:
from src.predict import predict
sample = integrated.sample(5, random_state=1)
pd.DataFrame({"customer_id": sample["customer_id"], "really_churned": sample[TARGET],
              "churn_probability": predict(sample).round(3)})

## 8. Power BI export
`python -m src.powerbi` writes the gold table as a small star schema (one fact table and
dimension tables) into `results/powerbi/`. `results/powerbi/POWERBI_GUIDE.md` explains how to
build the Power BI report from it.

In [ ]:
from src import powerbi
powerbi.run()